In [1]:
import pandas as pd
import requests
import os
import json
import time
import random

## Получаем даынне о зданиях, которые расположенны в радиусе 300 метров для каждой странции

- читаю файл csv bike_share/station_list.csv 

In [2]:
station_list = pd.read_csv('../../data_additional/station_list.scv')
station_list

,station_id,station_name,start_lat,start_lng
0,30200,9th St & Pennsylvania Ave NW,38.894280,-77.023979
1,30201,9th & G St NW,38.898097,-77.023924
2,31000,Eads St & 15th St S,38.858971,-77.053230
3,31001,18th St & S Eads St,38.857250,-77.053320
4,31002,Crystal Dr & 20th St S,38.856425,-77.049232
...,...,...,...,...
908,32608,Falls Church City Hall / Park Ave & Little Fal...,38.885434,-77.173605
909,32609,W Columbia St & N Washington St,38.885621,-77.166917
910,32900,Motivate BX Tech office,38.964406,-77.010759
911,32901,6035 Warehouse,38.963810,-77.010266


- заменяю '/' на ''  в столбце station_name, что бы записывать потом в json файл

In [3]:
station_list['station_name'] = station_list['station_name'].apply(lambda x: x.replace('/', '') if '/' in x else x)

- написал функцию 

In [4]:
def get_data(start_lat, start_lng, distance=300) -> requests.Response:
    """
    Получает данные из OpenStreetMap через Overpass API.

    :param start_lat: широта
    :param start_lng: долгота
    :param distance: радиус поиска в метрах
    :return: Response
    """

    query = f"""
    [out:json][timeout:60];
    node["amenity"](around:{distance},{start_lat},{start_lng});
    out body;
    """

    url = "https://overpass-api.de/api/interpreter"

    headers = {
        "User-Agent": "my-overpass-script/1.0",
        "Accept": "application/json",
        "Content-Type": "text/plain"
    }

    response = requests.post(
        url=url,
        data=query,
        headers=headers,
        timeout=120
    )

    return response

- написал скрипт что бы собрать данные (пришлось делать много обработок так как overpass не может выдеражать большую нагрузку)

In [5]:
for row in station_list.itertuples(index=False):
    station_id = row.station_id
    station_name = row.station_name
    lat = row.start_lat
    lng = row.start_lng

    success = False

    for attempt in range(5):
        try:
            response = get_data(lat, lng)

            if response.status_code == 200:
                data = response.json()

                file_path = os.path.join('../../overpass_data/data_about_build', f"{station_id}_{station_name}.json")

                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                print(station_id, "✅ saved")
                success = True
                break

            elif response.status_code == 429:
                sleep_time = 20 * (attempt + 1)
                print(station_id, "⚠ 429, sleep", sleep_time)
                time.sleep(sleep_time)

            else:
                print(station_id, "error:", response.status_code)
                time.sleep(5)

        except Exception as e:
            print(station_id, "exception:", e)
            time.sleep(10)

    if not success:
        print(station_id, "❌ failed after 5 attempts")

    # мягкий рандомный sleep между станциями
    time.sleep(random.uniform(2, 5))

30200 ✅ saved
30201 error: 504
30201 ✅ saved
31000 ✅ saved
31001 ⚠ 429, sleep 20
31001 error: 504
31001 ✅ saved
31002 ✅ saved
31003 ✅ saved
31004 ✅ saved
31005 ⚠ 429, sleep 20
31005 error: 504
31005 ✅ saved
31006 ✅ saved
31006 ✅ saved
31007 ⚠ 429, sleep 20
31007 ✅ saved
31009 ✅ saved
31010 ✅ saved
31011 ✅ saved
31012 ⚠ 429, sleep 20
31012 ✅ saved
31013 ✅ saved
31014 ⚠ 429, sleep 20
31014 ✅ saved
31014 error: 504
31014 exception: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))
31014 ✅ saved
31015 ✅ saved
31015 ✅ saved
31016 ✅ saved
31017 ⚠ 429, sleep 20
31017 ✅ saved
31018 ✅ saved
31019 exception: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol 

- нужно посмотреть какие станции не загрузились